## Side-by-side timeseries animations for Land Cover 2.0 and Geomedian

This notebook uses the [Land Cover 3.0](https://knowledge.dea.ga.gov.au/data/product/dea-land-cover-landsat/) and [Geometric Median and Median Absolute Deviation (Landsat)](https://knowledge.dea.ga.gov.au/data/product/dea-geometric-median-and-median-absolute-deviation-landsat/) products to create animations showing areas of interest throughout time in yearly timesteps.

This notebook connects to the development sandbox database currently; once Land Cover 3.0 is publicly accessible this can be updated and access to the database replaced with use of the `datacube` python package for accessing the datasets.

The notebook has a cell for the user to input parameters. These will then be passed to functions that will connect to the database, generate individual animations, and then generate paired animations. These will be saved out to the directory specified in the parameter cell.


In [1]:
%matplotlib inline

import os
import sys
import pandas as pd
import numpy as np
import datacube
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patheffects as PathEffects
from datacube.utils.masking import make_mask
from datacube.virtual import catalog_from_file
from datacube.virtual import construct_from_yaml
from matplotlib import colors as mcolours
from IPython.display import Image
from IPython.core.display import Video
from shapely.geometry import shape, box
from skimage.exposure import rescale_intensity
from pathlib import Path

from PIL import ImageSequence
from PIL import Image as pimg

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import xr_animation
from dea_tools.spatial import xr_vectorize, add_geobox
from dea_tools.landcover import lc_animation, lc_colourmap
from dea_tools.dask import create_local_dask_cluster
from dea_tools.datahandling import load_ard

The cell below is only required if trying to access data that is only in the development database. If you do not need to do that, you can comment out the cell and proceed using `datacube`

In [2]:
client = create_local_dask_cluster(return_client=True)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41085,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:44111,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/38255/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:46501,


In [3]:
dc = datacube.Datacube(app="geomad_aminations")

In [4]:
"""
Use the boolean flags to turn on/off sections of the notebook. If you only want to create the masked animations for example, turn core_animations and lonterm_animations to False.
This speeds up the notebook.

To run this notebook, you should supply a csv with each area you are interested stored as a row.

TODO: Include a template CSV with notebooks when work is all done.
"""
core_animations = False
masked_animations = False
longterm_animations = False

output_dir = 'output_gifs'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

text_size = 35
dpi = 150

In [5]:
# The '_slim' csv contains a minimal set of testing locations. Swap for a complete csv for final testing.

df = pd.read_csv('input_csv/input_for_missing_tiles_timeseries.csv')

lats = df['centre_y'].tolist()
lons = df['centre_x'].tolist()
buffers = [i/2 for i in df['buffer_size_wgs84']]
times = df.apply(lambda row: (str(row['start_year']), str(row['end_year'])), axis=1).tolist()
intervals = df['interval'].tolist()
roi_names = df['name'].tolist()

In [6]:
df

,centre_x,centre_y,buffer_size_wgs84,start_year,end_year,interval,name,level,level3_class,level4_class,comment
0,126.853,-21.652,6,1988,2023,400,missing_tiles_WA,4,NaN,NaN,buffer area just containing the 3 missing tiles
1,126.853,-21.652,20,1988,2023,400,zoom_out_missing_tiles_WA,4,NaN,NaN,larger buffer area to include northern coast o...


In [7]:
def get_normalized_rgb(class_number):
    colours = data["lc_colours"]["level3"].get(str(class_number))
    if colours:
        return tuple(colour / 255 for colour in colours[:3])
    else:
        return None

In [8]:
def xr_animation_modified(ds,
                 bands=None,
                 output_path='animation.mp4',
                 width_pixels=500,
                 interval=100,
                 percentile_stretch=(0.02, 0.98),
                 image_proc_funcs=None,
                 show_gdf=None,
                 show_date='%d %b %Y',
                 show_text=None,
                 show_colorbar=True,
                 gdf_kwargs={},
                 annotation_kwargs={},
                 imshow_kwargs={},
                 colorbar_kwargs={},
                 limit=None,
                 list_extra_labels=[]):

    # this is a modified version of xr_animation that allows us to add text to animations. See original xr_animation function for more documentation.
    def _start_end_times(gdf, ds):
        # Make copy of gdf so we do not modify original data
        gdf = gdf.copy()

        # Get min and max times from input dataset
        minmax_times = pd.to_datetime(ds.time.isel(time=[0, -1]).values)

        # Update both `start_time` and `end_time` columns
        for time_col, time_val in zip(['start_time', 'end_time'], minmax_times):

            # Add time_col if it does not exist
            if time_col not in gdf:
                gdf[time_col] = np.nan

            # Convert values to datetimes and fill gaps with relevant time value
            gdf[time_col] = pd.to_datetime(gdf[time_col], errors='ignore')
            gdf[time_col] = gdf[time_col].fillna(time_val)

        return gdf

    def _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults):
        # Create new axis object for colorbar
        cax = fig.add_axes([0.02, 0.02, 0.96, 0.03])

        # Initialise color bar using plot min and max values
        img = ax.imshow(np.array([[vmin, vmax]]), **imshow_defaults)
        fig.colorbar(img,
                     cax=cax,
                     orientation='horizontal',
                     ticks=np.linspace(vmin, vmax, 2))

        # Fine-tune appearance of colorbar
        cax.xaxis.set_ticks_position('top')
        cax.tick_params(axis='x', **colorbar_defaults)
        cax.get_xticklabels()[0].set_horizontalalignment('left')
        cax.get_xticklabels()[-1].set_horizontalalignment('right')

    def _frame_annotation(times, show_date, show_text):
        # Test if show_text is supplied as a list
        is_sequence = isinstance(show_text, (list, tuple, np.ndarray))

        # Raise exception if it is shorter than number of dates
        if is_sequence and (len(show_text) == 1):
            show_text, is_sequence = show_text[0], False
        elif is_sequence and (len(show_text) < len(times)):
            raise ValueError(f'Annotations supplied via `show_text` must have '
                             f'either a length of 1, or a length >= the number '
                             f'of timesteps in `ds` (n={len(times)})')

        times_list = (times.dt.strftime(show_date).values
                      if show_date else [None] * len(times))
        text_list = show_text if is_sequence else [show_text] * len(times)
        annotation_list = [
            '\n'.join([str(i)
                       for i in (a, b)
                       if i])
            for a, b in zip(times_list, text_list)
        ]

        return annotation_list

    def _update_frames(i, ax, extent, annotation_text, gdf, gdf_defaults,
                       annotation_defaults, imshow_defaults):

        # Clear previous frame to optimise render speed and plot imagery
        ax.clear()
        ax.imshow(array[i, ...].clip(0.0, 1.0),
                  extent=extent,
                  vmin=0.0,
                  vmax=1.0,
                  **imshow_defaults)

        # Add annotation text
        ax.annotate(annotation_text[i], **annotation_defaults)

        # Add geodataframe annotation
        if show_gdf is not None:

            # Obtain start and end times to filter geodataframe features
            time_i = ds.time.isel(time=i).values

            # Subset geodataframe using start and end dates
            gdf_subset = show_gdf.loc[(show_gdf.start_time <= time_i) &
                                      (show_gdf.end_time >= time_i)]

            if len(gdf_subset.index) > 0:

                # Set color to geodataframe field if supplied
                if ('color' in gdf_subset) and ('color' not in gdf_kwargs):
                    gdf_defaults.update({'color': gdf_subset['color'].tolist()})

                gdf_subset.plot(ax=ax, **gdf_defaults)

        # Remove axes to show imagery only
        ax.axis('off')


    # Add GeoBox and odc.* accessor to array using `odc-geo`
    try:
        ds = add_geobox(ds)
    except ValueError:
        raise ValueError("Unable to determine `ds`'s coordinate "
                         "reference system (CRS). Please assign a CRS "
                         "to the array before passing it to this "
                         "function, e.g.: "
                         "`ds.odc.assign_crs(crs='EPSG:3577')`")
    
    # Test if bands have been supplied, or convert to list to allow
    # iteration if a single band is provided as a string
    if bands is None:
        raise ValueError(f'Please use the `bands` parameter to supply '
                         f'a list of one or three bands that exist as '
                         f'variables in `ds`, e.g. {list(ds.data_vars)}')
    elif isinstance(bands, str):
        bands = [bands]

    # Test if bands exist in dataset
    missing_bands = [b for b in bands if b not in ds.data_vars]
    if missing_bands:
        raise ValueError(f'Band(s) {missing_bands} do not exist as '
                         f'variables in `ds` {list(ds.data_vars)}')

    # Test if time dimension exists in dataset
    if 'time' not in ds.dims:
        raise ValueError(f"`ds` does not contain a 'time' dimension "
                         f"required for generating an animation")

    # Set default parameters
    outline = [PathEffects.withStroke(linewidth=2.5, foreground='black')]
    annotation_defaults = {
        'xy': (1, 1),
        'xycoords': 'axes fraction',
        'xytext': (-5, -5),
        'textcoords': 'offset points',
        'horizontalalignment': 'right',
        'verticalalignment': 'top',
        'fontsize': 20,
        'color': 'white',
        'path_effects': outline
    }
    imshow_defaults = {'cmap': 'magma', 'interpolation': 'nearest'}
    colorbar_defaults = {'colors': 'white', 'labelsize': 12, 'length': 0}
    gdf_defaults = {'linewidth': 1.5}

    # Update defaults with kwargs
    annotation_defaults.update(annotation_kwargs)
    imshow_defaults.update(imshow_kwargs)
    colorbar_defaults.update(colorbar_kwargs)
    gdf_defaults.update(gdf_kwargs)

    # Get info on dataset dimensions
    height, width = ds.odc.geobox.shape
    scale = width_pixels / width
    left, bottom, right, top = ds.odc.geobox.extent.boundingbox

    # Prepare annotations
    annotation_list = _frame_annotation(ds.time, show_date, show_text)

    if len(list_extra_labels)>0: # if a list of extra labels is provided
        annotation_list = [f'{a}\n{b}' for a,b in zip(annotation_list, list_extra_labels)]

    # Prepare geodataframe
    if show_gdf is not None:
        show_gdf = show_gdf.to_crs(ds.odc.geobox.crs)
        show_gdf = gpd.clip(show_gdf, mask=box(
            left, bottom, right, top)).reindex(show_gdf.index).dropna(how='all')
        show_gdf = _start_end_times(show_gdf, ds)

    # Convert data to 4D numpy array of shape [time, y, x, bands]
    ds = ds[bands].to_array().transpose(..., 'variable')[0:limit, ...]
    array = ds.astype(np.float32).values

    # Optionally apply image processing along axis 0 (e.g. to each timestep)
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} ({remaining_s:.1f} ' \
                   'seconds remaining at {rate_fmt}{postfix})'
    if image_proc_funcs:
        print('Applying custom image processing functions')
        for i, array_i in tqdm(enumerate(array),
                               total=len(ds.time),
                               leave=False,
                               bar_format=bar_format,
                               unit=' frames'):
            for func in image_proc_funcs:
                array_i = func(array_i)
            array[i, ...] = array_i

    # Clip to percentiles and rescale between 0.0 and 1.0 for plotting
    vmin, vmax = np.quantile(array[np.isfinite(array)], q=percentile_stretch)

    # Replace with vmin and vmax if present in `imshow_defaults`
    if 'vmin' in imshow_defaults:
        vmin = imshow_defaults.pop('vmin')
    if 'vmax' in imshow_defaults:
        vmax = imshow_defaults.pop('vmax')

    # Rescale between 0 and 1
    array = rescale_intensity(array,
                              in_range=(vmin, vmax),
                              out_range=(0.0, 1.0))
    array = np.squeeze(array)  # remove final axis if only one band

    # Set up figure
    fig, ax = plt.subplots()
    fig.set_size_inches(width * scale / 72, height * scale / 72, forward=True)
    fig.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)

    # Optionally add colorbar
    if show_colorbar & (len(bands) == 1):
        _add_colorbar(fig, ax, vmin, vmax, imshow_defaults, colorbar_defaults)

    # Animate
    print(f'Exporting animation to {output_path}')
    anim = FuncAnimation(
        fig=fig,
        func=_update_frames,
        fargs=(
            ax,  # axis to plot into
            [left, right, bottom, top],  # imshow extent
            annotation_list,  # list of text annotations
            show_gdf,  # geodataframe to plot over imagery
            gdf_defaults,  # any kwargs used to plot gdf
            annotation_defaults,  # kwargs for annotations
            imshow_defaults),  # kwargs for imshow
        frames=len(ds.time),
        interval=interval,
        repeat=False)



    # Export animation to file
    if Path(output_path).suffix == '.gif':
        anim.save(output_path, writer='pillow')
    else:
        anim.save(output_path, dpi=72)

In [9]:
def generate_single_geomad_animations(gmad_ds, roi_name, interval):
    """
    Generates GIF animations for land cover levels 3 and 4, and geomad data.

    Parameters:
    gmad_ds (xarray.Dataset): Dataset containing geomad data.
    roi_name (str): Name of the region of interest.
    interval (int): Interval between frames in the animation.

    Returns:
    tuple: A tuple containing:
        - file_name_gmad (str): File path of the generated geomad animation GIF.
    """
    file_name_gmad = f'{output_dir}/{roi_name}_timeseries_geomad.gif'

    #generate geomad animations
    xr_animation(ds=gmad_ds,
                bands=['nbart_red', 'nbart_green','nbart_blue'],
                output_path=f'{output_dir}/{roi_name}_timeseries_geomad.gif',
                interval=interval,
                width_pixels=350,
                show_colorbar=False,
                show_date = False,
                percentile_stretch=(0.02, 0.98),
                annotation_kwargs= {'fontsize': 25})
    plt.close()

    return file_name_gmad


In [10]:
def timeseries_animation(file_name_gmad, file_name_landcover, aspect_ratio=None, crop=False):
    """
    Creates a side-by-side GIF animation combining geomad and land cover animations, ensuring they are synchronized.

    Parameters:
    file_name_gmad (str): File path of the geomad animation GIF.
    file_name_landcover (str): File path of the land cover animation GIF.
    aspect_ratio (float): The desired aspect ratio to crop to (width/height).

    Returns:
    str: File path of the combined animation GIF.
    """
    final_animations_dir = os.path.join(output_dir, 'final_animations')
    if not os.path.exists(final_animations_dir):
        os.makedirs(final_animations_dir)

    directory, filename = os.path.split(file_name_landcover)
    name, ext = os.path.splitext(filename)
    new_filename = f"{name}_gmad{ext}"
    output_filepath = os.path.join(directory, 'final_animations', new_filename)

    gif_lc = pimg.open(file_name_landcover)
    gif_gmad = pimg.open(file_name_gmad)

    if crop is True:
        frames1 = [frame.crop(crop_box) for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.crop(crop_box) for frame in ImageSequence.Iterator(gif_lc)]

    else:
        frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif_lc)]
        

    # Get dimensions of cropped gifs
    frame_width, frame_height = frames1[0].size

    # Set the figure size dynamically based on the size of the gifs
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(frame_width * 2 / dpi, frame_height / dpi))

    # Function to make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')

    # Adjust layout to minimize white space
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

    # Animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=interval)  # Adjust interval as needed

    animation_object.save(output_filepath, writer="Pillow")
    plt.close()

    return output_filepath

In [11]:
lat = lats[0]
lon = lons[0]
buffer = 4
interval = 600

lat_range = (lat - buffer, lat + buffer)
lon_range = (lon - buffer, lon + buffer)
    
query ={
    'x': lon_range,
    'y': lat_range,
    'time': ('1988', '2023'),
    'resolution': (-150, 150),
    'resampling': 'cubic',
    'group_by': 'solar_day'
}

In [12]:
products = ['ga_ls5t_gm_cyear_3', 'ga_ls7e_gm_cyear_3', 'ga_ls8cls9c_gm_cyear_3']

ds = dc.load(product=products,
            measurements = ['nbart_blue', 'nbart_green', 'nbart_red', 'count'],
            dask_chunks = {'time':2, 'x':2048, 'y':2048},
            **query)

In [13]:
ds

<xarray.Dataset> Size: 10GB
Dimensions:      (time: 36, y: 6160, x: 5713)
Coordinates:
  * time         (time) datetime64[ns] 288B 1988-07-01T23:59:59.999999 ... 20...
  * y            (y) float64 49kB -1.884e+06 -1.884e+06 ... -2.808e+06
  * x            (x) float64 46kB -9.706e+05 -9.704e+05 ... -1.138e+05
    spatial_ref  int32 4B 3577
Data variables:
    nbart_blue   (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    nbart_green  (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    nbart_red    (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    count        (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [14]:
# replace 0 with np.nan in the count variable, so that the cmap isn't applied to missing data.
ds['count'] = ds['count'].where(ds['count'] != 0, np.nan)

In [15]:
ds

<xarray.Dataset> Size: 18GB
Dimensions:      (time: 36, y: 6160, x: 5713)
Coordinates:
  * time         (time) datetime64[ns] 288B 1988-07-01T23:59:59.999999 ... 20...
  * y            (y) float64 49kB -1.884e+06 -1.884e+06 ... -2.808e+06
  * x            (x) float64 46kB -9.706e+05 -9.704e+05 ... -1.138e+05
    spatial_ref  int32 4B 3577
Data variables:
    nbart_blue   (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    nbart_green  (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    nbart_red    (time, y, x) int16 3GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
    count        (time, y, x) float64 10GB dask.array<chunksize=(2, 2048, 2048), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [16]:
# # # animate the geomad count and true colour
roi_name = 'gqa_tiles'

gmad_count_fpath = f'{output_dir}/{roi_name}_timeseries_geomad_obs_count.gif'
gmad_rgb_fpath = f'{output_dir}/{roi_name}_timeseries_geomad_rgb.gif'

xr_animation(ds=ds,
            bands=['count'],
            output_path=gmad_count_fpath,
            interval=interval,
            width_pixels=350,
            show_colorbar=False,
            show_date = '%Y',                                                                                                                                                                                                                               
            percentile_stretch=(0.02, 0.98),
            annotation_kwargs= {'fontsize': 25},
            imshow_kwargs= {'cmap': 'Spectral', 'vmin':1, 'vmax':50})
plt.close()

/env/lib/python3.10/site-packages/distributed/client.py:3361: UserWarning: Sending large graph of size 16.95 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Exporting animation to output_gifs/gqa_tiles_timeseries_geomad_obs_count.gif


  0%|          | 0/36 (0.0 seconds remaining at ? frames/s)

In [23]:
xr_animation(ds=ds,
            bands=['nbart_red', 'nbart_green', 'nbart_blue'],
            output_path=gmad_rgb_fpath,
            interval=interval,
            width_pixels=350,
            show_colorbar=False,
            show_date = '%Y',                                                                                                                                                                                                                               
            percentile_stretch=(0.01, 0.95),
            annotation_kwargs= {'fontsize': 25})
plt.close()

/env/lib/python3.10/site-packages/distributed/client.py:3361: UserWarning: Sending large graph of size 17.73 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Exporting animation to output_gifs/gqa_tiles_timeseries_geomad_rgb.gif


  0%|          | 0/36 (0.0 seconds remaining at ? frames/s)

In [18]:
gif_for_crop = pimg.open(gmad_rgb_fpath)

width, height = gif_for_crop.size

# Define the crop box based on the width and height
# For example, cropping 50 pixels from each side
left = 10
upper = 0
right = width
lower = height - 15
crop_box = (left, upper, right, lower)

In [24]:
timeseries_animation(gmad_rgb_fpath, gmad_count_fpath, aspect_ratio=None, crop=True)

MovieWriter Pillow unavailable; using Pillow instead.


'output_gifs/final_animations/gqa_tiles_timeseries_geomad_obs_count_gmad.gif'